# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vaib-raksh/Intern-ML/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### Distributions- answer

I examined the distributions of the main Google Search Console metrics used in my lane: `gsc_impressions`, `gsc_clicks`, and `gsc_sum_position`.

The summary statistics show that these variables are heavily right-skewed (heavy-tailed). For example, the median number of impressions is only **16**, while the maximum is **40,084**. Similarly, at least half of the webpages receive **0 clicks**, but a few pages receive as many as **274 clicks**. This indicates that most webpages receive relatively little traffic, while a small number account for a large share of impressions and clicks.

Understanding these distributions is important before testing relationships because averages alone may not represent the typical webpage.

In [1]:
!pip install -q duckdb datasets huggingface_hub   #connecting the dataset- install the required libraries

In [2]:
# Read the HF_TOKEN from colab
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded successfully!" if HF_TOKEN else "Token not found")

#connect DuckDB to Hugging face
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
);
""")

print("Connected successfully!")


Token loaded successfully!
Connected successfully!


In [3]:
DATASET = "hf://datasets/FlyRank/internship-warehouse" # the dataset path

In [4]:
distribution_df = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position
FROM read_parquet(
    '{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
""").df()

distribution_df.describe()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_impressions,gsc_clicks,gsc_sum_position
count,3.611061e+06,3.611061e+06,3.611061e+06
mean,7.772164e+01,2.275874e-01,8.990203e+02
std,2.498747e+02,1.277267e+00,3.829981e+03
min,1.000000e+00,0.000000e+00,0.000000e+00
25%,4.000000e+00,0.000000e+00,3.400000e+01
50%,1.600000e+01,0.000000e+00,1.600000e+02
75%,6.200000e+01,0.000000e+00,5.440000e+02
max,4.008400e+04,2.740000e+02,4.819460e+05


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal Test 1

**Hypothesis:** Pages with better search visibility have higher CTR.

**Test:** I divided webpages into three visibility groups using `NTILE(3)` based on `gsc_sum_position` and calculated the average CTR for each group.

**Result:** High Visibility = 0.004326, Medium Visibility = 0.002483, Low Visibility = 0.002433.

**Verdict:** **CONFIRMED.** The observed data shows that pages in higher visibility groups have higher average CTR than pages in lower visibility groups.

In [14]:
signal1 = con.sql(f"""
WITH position_groups AS (

SELECT
    gsc_clicks,
    gsc_impressions,
    gsc_sum_position,

    NTILE(3) OVER (
        ORDER BY gsc_sum_position
    ) AS position_group

FROM read_parquet(
'{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE
gsc_data_available IS TRUE
)

SELECT

CASE
    WHEN position_group = 1 THEN 'High Visibility'
    WHEN position_group = 2 THEN 'Medium Visibility'
    ELSE 'Low Visibility'
END AS visibility_group,

AVG(
CAST(gsc_clicks AS DOUBLE) /
NULLIF(gsc_impressions,0)
) AS avg_ctr,

COUNT(*) AS pages

FROM position_groups

GROUP BY position_group

ORDER BY position_group

""").df()

signal1


,visibility_group,avg_ctr,pages
0,High Visibility,0.004326,1203687
1,Medium Visibility,0.002483,1203687
2,Low Visibility,0.002433,1203687


In [7]:
signal2 = con.sql(f"""
SELECT

AVG(gsc_impressions) AS avg_impressions,
AVG(gsc_clicks) AS avg_clicks

FROM read_parquet(
'{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE gsc_data_available IS TRUE
""").df()

signal2

,avg_impressions,avg_clicks
0,77.721642,0.227587


In [8]:
signal3 = con.sql(f"""
SELECT

client_has_ga4,

AVG(gsc_clicks) AS avg_clicks,

COUNT(*) AS pages

FROM read_parquet(
'{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE gsc_data_available IS TRUE

GROUP BY client_has_ga4
""").df()

signal3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_has_ga4,avg_clicks,pages
0,False,0.219009,1528366
1,True,0.233883,2082695


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.